In [1]:
import math

### RCS Estimates of J20 and F35

In [2]:
J20 = {
    1 : {
        'median':0.24,
        'mean':0.32
    },
    3 : {
        'median':0.21,
        'mean':0.27
    },
    8 : {
        'median':0.21,
        'mean':0.28
    },
}

F35 = {
    1 : {
        'median':0.13,
        'mean':0.27
    },
    3 : {
        'median':0.08,
        'mean':0.15
    },
    8 : {
        'median':0.06,
        'mean':0.09
    },
}

In [3]:
def m2_to_db(m2):
    return 10*math.log10(m2)

def db_to_m2(db):
    return (10**(db/10))    ##return (10**db)**(1/10)


db_to_m2(-23)

0.005011872336272725

### RCS estimate at 2GhZ

In [61]:
#F22 at 2Ghz measurement
F22db_sband_2ghz_min = -9
F22db_sband_2ghz_max = -5.804921066

F22m2_sband_2ghz_min = db_to_m2(F22db_sband_2ghz_min)
F22m2_sband_2ghz_max = db_to_m2(F22db_sband_2ghz_max)

print("F22 mean at 2ghz", F22m2_sband_2ghz_min, '<->', F22m2_sband_2ghz_max)

F22 mean at 2ghz 0.12589254117941673 <-> 0.26272892776091283


In [62]:
#F35 measurement at 2 GHZ

F35_median_1gz = F35[1]['median']
F35_median_3gz = F35[3]['median']
F35_median_8gz = F35[8]['median']

F35_mean_1ghz = F35[1]['mean']
F35_mean_3ghz = F35[3]['mean']
F35_mean_8ghz = F35[8]['mean']

for ghz in F35.keys():
    print(f"{ghz}ghz: {F35[ghz]['mean']} m2 mean")
    print(f"{ghz}ghz: {F35[ghz]['median']} m2 median")

1ghz: 0.27 m2 mean
1ghz: 0.13 m2 median
3ghz: 0.15 m2 mean
3ghz: 0.08 m2 median
8ghz: 0.09 m2 mean
8ghz: 0.06 m2 median


##### Calculate surface absorption factor

In [63]:
# average between median and mean so that the spikes matters

f35_1_ghz = (F35[1]['mean'] + F35[1]['median'])/2
f35_3_ghz = (F35[3]['mean'] + F35[3]['median'])/2
f35_8_ghz = (F35[8]['mean'] + F35[8]['median'])/2

In [64]:
f35_1_ghz = F35[1]['mean']
f35_3_ghz = F35[3]['mean']
f35_8_ghz = F35[8]['mean']

In [65]:
# Calculate absorption factor from 1 ghz to 3 ghz

f1_f35_rcs_1ghz = f35_1_ghz
f2_f35_rcs_3ghz = f35_3_ghz

f1 = 1 # GHZ
f2 = 3 # GHZ

A1 = (math.log(f2_f35_rcs_3ghz/f1_f35_rcs_1ghz) / math.log(f2/f1)) - 2

A1

-2.535026479282073

In [66]:
f1_f35_rcs_3_ghz = f35_3_ghz
f2_f35_rcs_8_ghz = f35_8_ghz

print(f1_f35_rcs_3_ghz)
print(f2_f35_rcs_8_ghz)

0.15
0.09


In [67]:
# Calculate absorption factor from 3 ghz to 8 ghz


f1 = 3 # GHZ
f2 = 8 # GHZ

A2 = (math.log(f2_f35_rcs_8_ghz/f1_f35_rcs_3_ghz) / math.log(f2/f1)) - 2

A2

-2.5208099393420964

### RCS conversion to 8GhZ

In [68]:
# Assuming we use the same base composites, A remains the same
# Calculate F-22 rcs at 3 ghz

f1_f22_rcs = db_to_m2(F22db_sband_2ghz_max)


f1 = 2 # GHZ
f2 = 3 # GHZ

f2_f22_rcs = f1_f22_rcs * (f2/f1) ** (2 + A1)

f2_f22_rcs

0.21149222539763127

In [69]:
# Calculate F-22 RCS at 8 ghz

f1_f22_rcs = f2_f22_rcs


f1 = 3 # GHZ
f2 = 8 # GHZ

f2_f22_rcs = f1_f22_rcs * (f2/f1) ** (2 + A2)

f2_f22_rcs

0.12689533523857874

In [70]:
selected_F35_rcs = f35_8_ghz
selected_J20_rcs = 0.24
selected_F22_rcs = f2_f22_rcs

print(f"Selected F22 RCS: {selected_F22_rcs}")

Selected F22 RCS: 0.12689533523857874


In [71]:
print("J20 RCS in Dbm2:", m2_to_db(selected_J20_rcs))
print("F22 RCS in Dbm2:", m2_to_db(selected_F22_rcs))

J20 RCS in Dbm2: -6.19788758288394
F22 RCS in Dbm2: -8.965543425812651


In [72]:
##apply RAM coating
RAM_reduction_factor = 25


J20_rcs_m2 = db_to_m2(m2_to_db(selected_J20_rcs) - RAM_reduction_factor) 
F22_rcs_m2 = db_to_m2(m2_to_db(selected_F22_rcs) - RAM_reduction_factor)
F35_rcs_m2 = db_to_m2(m2_to_db(selected_F35_rcs) - RAM_reduction_factor)

print(f"J20 RCS: {J20_rcs_m2}\nF22 RCS: {F22_rcs_m2}\nF35 RCS: {F35_rcs_m2}")

J20 RCS: 0.0007589466384404108
F22 RCS: 0.0004012782838045344
F35 RCS: 0.00028460498941515434


In [73]:
'''
We know that AN/APG-81 has 1,676 GaAs T/R modules and that it's most likely the most advanced fighter radar ever. 
AN/APG-77(v)1 likely has somewhat longer detection/tracking ranges but the difference is not huge if we assume very 
similar components and layout. Radar range equations give about 15 percent difference in favour of AN/APG-77v(1). 
AN/APG-81 likely is superior in many other aspects (A/G, EW, communications) as it's newer design from the same company.

I think 150 km detection range for 1 m^2 target is way too short for AN/APG-81 max detection range. Older AN/APG-79 for 
Super Hornet is said to have 2-3 times longer ranges compared to AN/APG-73 used in Block 1 Super Hornets and that should 
compute to roughly 200-300 km detection range against 1 m^2 target. I'd say it's likely that AN/APG-81 has max detection 
range of well over 250 km against 1 m^2 target and I would not be surprised if it could reach 400 km in cued search. 
Of course the range depends on how large part of the sky is scanned and with large search areas the range is lower as the 
radar can't use a lot of time to scan any one part. Also tracking large number of simultaneous targets could also affect 
the search range. Detection range numbers differ greatly depending on many variables and can easily lead to apples to 
oranges type comparison.

https://www.f-16.net/forum/viewtopic.php?f=22&t=27750

Given a greater range of APG-77v1 radars compared to APG-81 radars, as well as wild reports ranging anywhere from a
conservative 160 km to 500+ km, I think 300 km is a reasonable estimate for the max detection range of a 1 m2 target

'''

def GetDetectionTrackingDistance(known_target_detection_distance, known_target_size, tracking_factor, query_target_size):
    detection_distance = known_target_detection_distance/(known_target_size/((query_target_size)**0.25))
    tracking_distance = detection_distance * tracking_factor

    return detection_distance, tracking_distance


In [74]:
'''
Assuming 400 km detection range of a 1m2 target by the APG-81

Assuming an 80% of detection range is tracking range, then the result is 1m2 at 320 km
'''

0.8 * 400

320.0

In [75]:
'''
APG 77 has a 1.15% range increase due to larger radome size
'''

368.1776*1.15

423.40423999999996

In [76]:

detection_target_size = 5**0.25

detection_range = 368.1776

km_detectable_by_apg77 = detection_range/(detection_target_size/((1)**0.25))


print(f"A radar that tracks a 5M^2 target at 368.1776 km can track a 1M^2 target at {km_detectable_by_apg77} and detect at {km_detectable_by_apg77/0.8}")

'''
- I'd say it's likely that AN/APG-81 has max detection 
range of well over 250 km against 1 m^2 target and I would not be surprised if it could reach 400 km in cued search. 

doubt: It has been said that the range has been increased to ~250nm (463km) vs the original APG-77 vs a 1m^2 target. To go from stated APG-81 ranges to more than that, is the question. We also don't know what the APG-77 MLU upgrades include.

doubt: The original APG-77 prior to upgrades to APG-77(v)1 already had a ~20% range advantage over the APG-81.

doubt: The original APG-77 had a ~12% longer range than the APG-81, prior to upgrading to the newer T/R modules. The APG-77(v)1 has a considerably longer range than the original APG-77, and the APG-81 hasn't gotten more powerful during that same period. 
When combined with much longer radar horizon, it gives the F-22 a much larger FOV. The F-22s radar is receiving further upgrades this decade, so it's long range advantage will likely remain, even after the F-35 gets the APG-85.

doubt: The original APG-77 had a 12% range advantage over the APG-81, prior to being upgraded. The APG-77(v)1 got a significant range increase over the original APG-77 (i.e. >50%)

'''

'''

- on GAN: The exact numbers are classified, but in general terms, the greater power-added efficiency of GaN compared with GaAs “roughly doubles the detection range with the same size, same aperture and the same amount of power,” Ditmars says.

- Part of the F-22 MLD is radar upgrades. It would be surprising if GaN modules weren't part of it, given that F-22s will be testbeds for emerging NGAD technology, and are the first jets to get AIM-260.

- Representing a major industry milestone in the advancement of the DoD’s mandated Modular Open Systems Approach (MOSA), Curtiss-Wright’s open standards-based processor card is the first COTS module to be selected for service 
onboard the F-22. The module will be used in support of the F-22 Tactical Mandates program to upgrade the aircraft’s Central Integrated Processor (CIP). The CIP provides data and signal processing for the F-22’s radar, sensors, 
electronic warfare, and other compute intensive capabilities.

- So I think this should also facilitate upgrade with APG-85 technology as CIP really is the back end of the APG-77. It seems like Curtiss-Wright uses the very best processors from Intel, AMD, NVidia, Xilinx etc, 
so processing power should not be a problem. 

- I also think that this APG-77v(2) (or whatever it will be called) will probably have even higher commonality with APG-85 than APG-77v(1) has with APG-81.

- There is an possible benefit of GaN for the AESA radars of both the F-22, F-35, or any other AESA radar.

-Actually increased output power can be very beneficial even for stealth aircraft if some thought is put into designing radar and other systems. Putting out more power does not directly mean the radar is more detectable. GaN technology gives many potential advantages that make it potentially much better than GaAs.

- Significantly higher output power of individual TR modules mean that it can create many more simultaneous beams or individual beams have higher power. There is no need to always blast at full power and modules can be run at low output power when high power is not needed. A lot of power is also good when a lot of power is needed. 
If enemy has VLO aircraft, having more radar power is better for detecting them and for jamming their radar.

- GaN allows much wider bandwidth which means it's much harder to detect and track. Current GaN systems can have 5-10 times the bandwidth of current GaAs systems. This means it can use 5-10 times more power and remain as tough to detect as GaAs system.

- GaN allows higher average power with the same peak power meaning that it will have better range performance while enemy ESM system performance stays the same as against GaAs system if all else remains equal.

Of course GaN also has other advantages like better efficiency. lower noise levels, smaller size and better reliability. Overall the improvement in performance can be as large as going from MSA or PESA radar to GaAs AESA. 
I'm sure F-35 will be the first mass produced fighter aircraft to adopt GaN radar system as that's where the development money really is and others are playing with pennies in comparison.

The reduced requirement of T&R modules in order to maintain the same radiating power means that the dish itself can be moved farther up into the nose. This allows the creation of "cheek" arrays on the sides of this new dish mount without having to do any changes to the body of the fighter.

On chinese GAN fielding:

But as we can see, some GaN fighter AESAs already exist in the west, and I wouldn't be surprised for more to emerge in the next few years as developmental cycles for projects begun in the recent past yield more radars.
I also wouldn't be surprised if the new AN/APG-85 for F-35 Block 4 is GaN.

And similarly for the PLA, I would expect radars that started development in the last 2-3 years to be GaN, including for fighters such as J-35/XY, J-20B/J-20S etc. But obviously for the PLA they're probably not going to tell us if they're GaN or not.

'''

detection_target_size = 1**0.25

detection_range = (350*0.8*1.15)*1.75

km_detectable_by_apg77 = detection_range/(detection_target_size/((F35_rcs_m2)**0.25))

print(f"F35 tracking range by f22 (km): {km_detectable_by_apg77}")

print("F35 tracking range by f22 (nm):",(km_detectable_by_apg77/1.852)*0.8)


detection_target_size = 1**0.25

detection_range = (350*0.8)*1.8

km_detectable_by_apg77 = detection_range/(detection_target_size/((F22_rcs_m2)**0.25))

print(f"F22 tracking range by F35 (km): {km_detectable_by_apg77}")

print("F22 tracking range by F35 (nm):",(km_detectable_by_apg77/1.852)*0.8)

detection_target_size = 1**0.25

detection_range = (350*0.8*1.15)*1.75

km_detectable_by_apg77 = detection_range/(detection_target_size/((J20_rcs_m2)**0.25))

print(f"J20 tracking range by f22 (km): {km_detectable_by_apg77}")

print("J20 tracking range by f22 (nm):",(km_detectable_by_apg77/1.852)*0.8)


'''In DCS, the F15's radar can detect a 5.5M^2 target at 68nm(125.9360 km)'''

detection_target_size = 1**0.25

detection_range = (463)*2

km_detectable_by_apg77 = detection_range/(detection_target_size/((F35_rcs_m2)**0.25))

##print(f"F22 tracking range by F15 (km): {km_detectable_by_apg77}")

##print("F22 tracking range by F15 (nm):",(km_detectable_by_apg77/1.852)*0.8)

##my estimate using 300km

##F35 detection range by f22 GAN=1.7 (nm): 35.76760173549717
##F22 detection range by F35 GAN=1.8 (nm): 31.79992480769475

##F35 detection range by f22 GAN=2 (nm): 42.07953145352608
##F22 detection range by F35 GAN=2 (nm): 35.3332497863275

## no GANs
##F35 detection range of 5^2 is 375
##F22 detection range of 5^2 is 431
##f22 detects F35 at 20.214100197701274
##F22 detects J20 at 26.03648220179761
##f35 detects f22 at 17.66662489316375
##F35 detects J20 at 23.358088099698815

## GANs = *2
##F22 detection range of J20 is 52.07296440359522
##F22 detection range of F35 is 39.20878831851106

##F35 detection range of J20 is 46.71617619939763
##F35 detection range of F22 is 35.3332497863275

## GAN = *1.8
##F22 detection range of J20 is 46.8656679632357
##F22 detection range of F35 is 35.287909486659956

##F35 detection range of J20 is 42.04455857945786
##F35 detection range of F22 is 31.79992480769475

## GAN Final
##F22 detection range of F35 is 33.3274700707344


##F35 detection range of F22 is 33.566587297011125

##F22 detection range of J20 is 44.262019743055944
##F35 detection range of J20 is 44.38036738942774

##J20 detection range of F22 is 33.47554878780983
##J20 detection range of F35 is 33.47554878780983

## F22 GAN=1.6, target size = 1m2, distance = 320km + 15%
## F35 GAN=1.85, target size = 1m2, distance = 320km
##F22 detection range of F35 is 43.874924795543194(35.09993983643456)
##F35 detection range of F22 is 41.70484659560708(33.36387727648567)



A radar that tracks a 5M^2 target at 368.1776 km can track a 1M^2 target at 246.21520050948712 and detect at 307.7690006368589
F35 tracking range by f22 (km): 73.19047197327122
F35 tracking range by f22 (nm): 31.6157546320826
F22 tracking range by F35 (km): 71.333240066821
F22 tracking range by F35 (nm): 30.81349462929633
J20 tracking range by f22 (km): 93.5290947277034
J20 tracking range by f22 (nm): 40.40133681542264


In [77]:

'''My take away from that is that the max instrumented range is 200nm. A big enough RCS will be seen from that far. The model I use for the APG-81 shows a 5m^2 RCS would be detected at 198.8nm, a 15M^2 target could be tracked at 203nm.'''

## from the pdf

target_size = 1**(0.25) ## known detection range of 1 m2 at 300 + 15% km
base_detection_distance = 220.388

array_size_factor = 1
GaN_factor = 1.85

tracking_factor = 1


apg81_measure_distance = base_detection_distance * 1 * GaN_factor

_J20_Detection_Distance,_  = GetDetectionTrackingDistance(apg81_measure_distance, target_size, tracking_factor, J20_rcs_m2)

print(f"J20 can be tracked at: {_J20_Detection_Distance/1.852} nm")


## from the comment

target_size = 5**(0.25) ## known detection range of 1 m2 at 300 + 15% km
base_detection_distance = 368.1776

array_size_factor = 1
GaN_factor = 1

tracking_factor = 1


apg81_measure_distance = base_detection_distance * 1 * GaN_factor

_1m2_tracking_distance,_  = GetDetectionTrackingDistance(apg81_measure_distance, target_size, tracking_factor, J20_rcs_m2)

print(f"A 1 m^2 target can be tracked at: {_1m2_tracking_distance/1.852} nm")

J20 can be tracked at: 36.540248809767355 nm
A 1 m^2 target can be tracked at: 22.066156266327873 nm


In [78]:
detection_target_size = 3**0.25

detection_range = (500*0.80*1.2)*1.8

km_detectable_by_apg77 = detection_range/(detection_target_size/((F35_rcs_m2)**0.25))

print(f"F35 tracking range by f22 (km): {km_detectable_by_apg77}")

print("F35 tracking range by f22 (nm):",(km_detectable_by_apg77/1.852)*0.8)


detection_target_size = 3**0.25

detection_range = (500*0.85)*1.8

km_detectable_by_apg77 = detection_range/(detection_target_size/((F22_rcs_m2)**0.25))

print(f"F22 tracking range by F35 (km): {km_detectable_by_apg77}")

print("F22 tracking range by F35 (nm):",(km_detectable_by_apg77/1.852)*0.8)

F35 tracking range by f22 (km): 85.26956671002256
F35 tracking range by f22 (nm): 36.83350613823869
F22 tracking range by F35 (km): 82.27019673131109
F22 tracking range by F35 (nm): 35.5378819573698


In [79]:
detection_target_size = 3**0.25

detection_range = (400*1*1)*1

km_detectable_by_apg77 = detection_range/(detection_target_size/((1)**0.25))

print(f"1 M^2 target detected at (km): {km_detectable_by_apg77}")

print("1 M^2 target detected at (nm):",(km_detectable_by_apg77/1.852))


1 M^2 target detected at (km): 303.93427426063704
1 M^2 target detected at (nm): 164.11137919040877


In [83]:
print(f"Chosen J20 RCS: {J20_rcs_m2}")
print(f"Chosen F22 RCS: {F22_rcs_m2}")
print(f"Chosen F35 RCS: {F35_rcs_m2}")

target_size = 1**(0.25) ## known detection range of 1 m2 at 300 + 15% km
base_detection_distance = 300

array_size_factor = 1.15
GaN_factor = 1.7

tracking_factor = 0.80

apg77_measure_distance = base_detection_distance * array_size_factor * GaN_factor
apg85_measure_distance = base_detection_distance * 1 * GaN_factor

print(f"AN/APG-77 measuring distance: {apg77_measure_distance}")

_, F35_by_F22 = GetDetectionTrackingDistance(apg77_measure_distance, target_size, tracking_factor, F35_rcs_m2)
_, J20_by_F22 = GetDetectionTrackingDistance(apg77_measure_distance, target_size, tracking_factor, J20_rcs_m2)

_, F22_by_F35 = GetDetectionTrackingDistance(apg85_measure_distance, target_size, tracking_factor, F22_rcs_m2)
_, J20_by_F35 = GetDetectionTrackingDistance(apg85_measure_distance, target_size, tracking_factor, J20_rcs_m2)

_, F22_by_J20 = GetDetectionTrackingDistance(apg77_measure_distance, target_size, tracking_factor, F22_rcs_m2)
_, F35_by_J20 = GetDetectionTrackingDistance(apg77_measure_distance, target_size, tracking_factor, F35_rcs_m2)

print(f"F35 tracked by F22 at: {F35_by_F22/1.852} nm ({F35_by_F22} km)")
print(f"F22 tracked by F35 at: {F22_by_F35/1.852} nm ({F22_by_F35} km)")
print("---------------------------------------------------------------")
print(f"J20 tracked by F35 at: {J20_by_F35/1.852} nm ({J20_by_F35} km)")
print(f"J20 tracked by F22 at: {J20_by_F22/1.852} nm ({J20_by_F22} km)")
print("---------------------------------------------------------------")
print(f"F35 tracked by J20 at: {F35_by_J20/1.852} nm ({F35_by_J20} km)")
print(f"F22 tracked by J20 at: {F22_by_J20/1.852} nm ({F22_by_J20} km)")


Chosen J20 RCS: 0.0007589466384404108
Chosen F22 RCS: 0.0004012782838045344
Chosen F35 RCS: 0.00028460498941515434
AN/APG-77 measuring distance: 586.5
F35 tracked by F22 at: 32.906193596657396 nm (60.9422705410095 km)
F22 tracked by F35 at: 31.180321946311764 nm (57.74595624456939 km)
---------------------------------------------------------------
J20 tracked by F35 at: 36.56553997491667 nm (67.71938003354568 km)
J20 tracked by F22 at: 42.05037097115417 nm (77.87728703857753 km)
---------------------------------------------------------------
F35 tracked by J20 at: 32.906193596657396 nm (60.9422705410095 km)
F22 tracked by J20 at: 35.85737023825853 nm (66.4078496812548 km)


## Adjust for air defence minimum RCS

##### 0.02 m^2 is the minimum RCS at which air defense of any radar power in DCS can pick up. This is a hard coded value that is irrespective of the search radar's emitting power or sensitivity

In [84]:
min_rcs = 0.02
adj_F35_RCS = 0.02
adj_F22_RCS = F22_rcs_m2 / F35_rcs_m2 * adj_F35_RCS
adj_J20_RCS = J20_rcs_m2 / F35_rcs_m2 * adj_F35_RCS

print(f"Adjusted F22 RCS: {adj_F22_RCS}")
print(f"Adjusted F35 RCS: {adj_F35_RCS}")
print(f"Adjusted J20 RCS: {adj_J20_RCS}")

Adjusted F22 RCS: 0.028198963386350778
Adjusted F35 RCS: 0.02
Adjusted J20 RCS: 0.05333333333333328


In [25]:
472042.21795918373 * 1.08# to determine the tracking distance of the J20 radar in DCS


509805.59539591847

In [26]:
337637.83/(86 / 77)

302303.63848837215

In [27]:
1525.581395348837/0.9

1695.0904392764855

In [28]:
450086.77/1.155


389685.51515151514

In [29]:
## min 7 nm difference (26 - 22.7)
max_detection_range_difference = 41.14553288135376 - 35.77872424465545

## max detection difference (27.9 - 18.53)
min_detection_range_difference = 27.9 - 18.53

print(min_detection_range_difference,'-',max_detection_range_difference)

radar_advantage = 3

print(min_detection_range_difference + radar_advantage,'-',max_detection_range_difference + radar_advantage)

9.369999999999997 - 5.366808636698309
12.369999999999997 - 8.366808636698309


In [30]:
27.942790857234005 - 18.513395373097055

9.42939548413695

In [31]:
db_to_m2(m2_to_db(0.07)-25)

0.0002213594362117866

In [32]:
27.942790857234005 * 0.85

23.751372228648904

In [33]:
'''#
note the F35's radar is scaled down from the F22 by 15% - according to some forum user:
Right now, the relationship between F-22 and J20 is modeled with a 3 nm radar advantage for the APG77v1 as having equal or more TR and being more dense.
The F35's detection range by the F-22 and of the F-22 isn't scaled properly. Instead, it's radar is scaled down from APG77v1 by 15%
The APG77 should detect the F35 at range 17nm, but instead does this at 21nm
It is unknown

If this is done properly, APG 77 should detect F35 at 17nm but then APG81 should detect F22 at an even shorter range. Both have comparable RCS but the F22 has a larger radar.

As it stands, 

the APG 81 detects the F22 at 17 nm 
the APG77 detects the F35 at 20.6 nm
the APG77 detects the J20 at roughly 27 nm
the J20 detects the F22 at roughly 21 nm
the J20 detects the F35 at roughly 20.2 nm

hornetfinn
    Elite 4K
    Elite 4K
     
    Posts: 4529
    Joined: Wed Mar 13, 2013 7:31 am
    Location: Finland

by hornetfinn » Mon Aug 17, 2015 12:45 pm
Numbers close to reality would require knowing the following: T/R module count, module output power, module duty cycle, module transmit losses, module receive losses, 
receiver sensitivity and radar level of digitization to name the biggest ones. We only know the module count and that gives us some clues but real world detection/tracking ranges we can only guess.

We know that AN/APG-81 has 1,676 GaAs T/R modules and that it's most likely the most advanced fighter radar ever. AN/APG-77(v)1 likely has somewhat longer detection/tracking ranges but the difference is not huge if we assume 
very similar components and layout. Radar range equations give about 15 percent difference in favour of AN/APG-77v(1). AN/APG-81 likely is 
superior in many other aspects (A/G, EW, communications) as it's newer design from the same company.

Comparison between two AESA radars made by different companies in different countries is impossible without knowing a lot of details about design and contruction. If PAK FA radar and AN/APG-81 radar had identical components and layout, AN/APG-81 would have slightly longer detection/tracking range. 
Larger size of PAK FA radar would not matter much and higher number of modules would make up for smaller size of AN/APG-81 antenna.
 However, they definitely do not have identical components and layout. US has such a huge lead in designing and manufacturing components for X-band AESA radars and similar integrated circuits that it's likely that AN/APG-81 has significantly longer effective ranges if similar values are compared (similar radar modes, similar targets, similar conditions). 
 How much longer is impossible to tell, but the difference can be substantial depending on exact details of each radar. 
 You have to remember that AN/APG-81 is fourth gen X-band AESA for US and third gen operational one. PAK FA will have first gen AESA for Russia.

I think 150 km detection range for 1 m^2 target is way too short for AN/APG-81 max detection range. Older AN/APG-79 for Super Hornet is said to have 2-3 times longer ranges compared to AN/APG-73 used in Block 1 Super Hornets and that should compute to roughly 200-300 km detection range against 1 m^2 target. 
I'd say it's likely that AN/APG-81 has max detection range of well over 250 km against 1 m^2 target and I would not be surprised if it could reach 400 km in cued search. Of course the range depends on how large part of the sky is scanned and with large search areas the range is lower as the radar can't use a lot of time to scan any one part. Also tracking large number of simultaneous targets could also affect the search range. Detection range numbers differ greatly depending on many variables and can easily lead to apples to oranges type comparison.


https://www.f-16.net/forum/viewtopic.php?f=33&t=5039&start=60
This new radar fitted to Lot 5 F-22s onwards and probably retrofitted to earlier lots is the AN/APG-77(v)1. It will use the new tile style T/R modules developed for the Superhornet's APG-79 and F-16 Blk 60's APG-80. The new T/R modules are smaller, more powerful and has better cooling properties. The radar may hence have more T/R elements while retaining the same size.

There is no mention of side arrays being included in the APG-77(v)1.

https://www.quora.com/Is-the-F22S-APG77V1-better-than-the-F35S-APG81

A lot depends on exact radar system in question and how good the whole system is. Especially the signal processing and tracker can vary a lot in capabilities. 
Especially so in look-down situations or when there is some other clutter or interference present.
 More modern radars can use different waveforms and other optimizations for different tasks and thus more modern radars usually have tracking range closer to detection range than old radars. 
 For old radars tracking range was usually something like 60-70 percent of detection range. For modern MSA or PESA, it is usually about 80 percent. AESA radars are more capable in their tracking capabilties. 
 They can for example use longer dwell time (put more power and computing effort) for tracked targets than for searching targets. AESA can have tracking range very similar to detection range because of that.

IRST systems are different in that their probability of detection goes from 0 to 100 percent very quickly when range decreases and back to 0 when range increases.
For IRST systems the tracking range can be very close to max detection range, naturally depending on how the tracker has been implemented. 
IRST systems are also different in that they don't measure range and also have much higher resolution and update interval, which help in this.

'''

"#\nnote the F35's radar is scaled down from the F22 by 15% - according to some forum user:\nRight now, the relationship between F-22 and J20 is modeled with a 3 nm radar advantage for the APG77v1 as having equal or more TR and being more dense.\nThe F35's detection range by the F-22 and of the F-22 isn't scaled properly. Instead, it's radar is scaled down from APG77v1 by 15%\nThe APG77 should detect the F35 at range 17nm, but instead does this at 21nm\nIt is unknown\n\nIf this is done properly, APG 77 should detect F35 at 17nm but then APG81 should detect F22 at an even shorter range. Both have comparable RCS but the F22 has a larger radar.\n\nAs it stands, \n\nthe APG 81 detects the F22 at 17 nm \nthe APG77 detects the F35 at 20.6 nm\nthe APG77 detects the J20 at roughly 27 nm\nthe J20 detects the F22 at roughly 21 nm\nthe J20 detects the F35 at roughly 20.2 nm\n\nhornetfinn\n    Elite 4K\n    Elite 4K\n     \n    Posts: 4529\n    Joined: Wed Mar 13, 2013 7:31 am\n    Location: Finla

In [34]:
'''
on the J-16B

I think we're back to square 1, unfortunately. The original poster exclaimed this was an L-band AESA, but I assumed he was just exaggerating. The 1760 module TR count in other documents means that this is not the J-11B AESA, 
it's something else, although presumably from these figures it's likely to be Flanker-sized. At the same time, though, the Chinese media claimed 450 km range on the J-11B AESA, so perhaps it IS L-band and the X-band AESA was discarded, or alternately the L-band achieves 450 km detection against 0 dBsm, but the X-band has an inferior detection range.

As for the translation,

Heading: Maximum Detection Range vs a RCS 0.1 m^2 target

Figure 4.5 Greatest Discovery Range at Dissimilar Elevations (RCS 0.1m^2)

Column Headers:

Target Elevation (Meters) / Maximum Detection Range In Normal Early Warning Mode (Kilometers) / Maximum View Line Range [???] (kilometers)

Paragraph:

Considering the F-22 stealth aircraft's RCS should be 0.4 m^2 [only makes sense if we're taking L-band, and I still have my doubts there], the radar's detection range against 0.4 m^2 is shown.

Second table:

Same as above.

'''

"\non the J-16B\n\nI think we're back to square 1, unfortunately. The original poster exclaimed this was an L-band AESA, but I assumed he was just exaggerating. The 1760 module TR count in other documents means that this is not the J-11B AESA, \nit's something else, although presumably from these figures it's likely to be Flanker-sized. At the same time, though, the Chinese media claimed 450 km range on the J-11B AESA, so perhaps it IS L-band and the X-band AESA was discarded, or alternately the L-band achieves 450 km detection against 0 dBsm, but the X-band has an inferior detection range.\n\nAs for the translation,\n\nHeading: Maximum Detection Range vs a RCS 0.1 m^2 target\n\nFigure 4.5 Greatest Discovery Range at Dissimilar Elevations (RCS 0.1m^2)\n\nColumn Headers:\n\nTarget Elevation (Meters) / Maximum Detection Range In Normal Early Warning Mode (Kilometers) / Maximum View Line Range [???] (kilometers)\n\nParagraph:\n\nConsidering the F-22 stealth aircraft's RCS should be 0.4 m^

In [35]:
db_to_m2(-4)

0.3981071705534972